# Medallion Architecture Query Notebook
Interactive queries for Bronze/Silver/Gold layers

In [2]:
# Initialize Spark with Delta Lake and S3 support
import os
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

# Get the downloaded JAR paths (same as other databricks scripts)
ivy_jars = os.path.expanduser("~/.ivy2/jars")
hadoop_aws_jar = f"{ivy_jars}/org.apache.hadoop_hadoop-aws-3.3.1.jar"
aws_sdk_jar = f"{ivy_jars}/com.amazonaws_aws-java-sdk-bundle-1.11.901.jar"
delta_spark_jar = f"{ivy_jars}/io.delta_delta-spark_2.12-3.2.1.jar"
delta_storage_jar = f"{ivy_jars}/io.delta_delta-storage-3.2.1.jar"

# Check if JARs exist
missing_jars = []
for jar_name, jar_path in [
    ("Hadoop AWS", hadoop_aws_jar),
    ("AWS SDK", aws_sdk_jar),
    ("Delta Spark", delta_spark_jar),
    ("Delta Storage", delta_storage_jar)
]:
    if not os.path.exists(jar_path):
        missing_jars.append(jar_name)

if missing_jars:
    print("❌ Missing JARs:", ", ".join(missing_jars))
    print(f"\n📦 To download JARs, run any databricks script first:")
    print("   python databricks/bronze/setup_bronze.py")
    raise FileNotFoundError(f"Required JARs not found in {ivy_jars}")

builder = SparkSession.builder \
    .appName("MedallionQuery") \
    .master("local[*]") \
    .config("spark.jars", f"{hadoop_aws_jar},{aws_sdk_jar},{delta_spark_jar},{delta_storage_jar}") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://localhost:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.secret.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")

spark = builder.getOrCreate()
print("✅ Spark session initialized with Delta Lake and S3 support")
print(f"📊 Spark UI: {spark.sparkContext.uiWebUrl}")
print(f"📦 Using JARs from: {ivy_jars}")

✅ Spark session initialized with Delta Lake and S3 support
📊 Spark UI: http://10.255.255.254:4040
📦 Using JARs from: /home/xuxoramos/.ivy2/jars


## 🥉 Bronze Layer - Raw Events

In [ ]:
# Verify MinIO connection and Delta Lake tables
import subprocess

# Check if MinIO is running
print("🔍 Checking MinIO container...")
result = subprocess.run(["docker", "ps", "--filter", "name=minio", "--format", "{{.Names}}"], 
                       capture_output=True, text=True)
if "minio" not in result.stdout:
    print("❌ MinIO container not running!")
    print("   Start it with: cd docker && docker-compose up -d")
    raise RuntimeError("MinIO container not running")
print("✅ MinIO container is running")

# Check which Delta Lake tables exist
print("\n📂 Checking Delta Lake tables:")
paths = {
    "Bronze (Events)": "s3a://lakehouse/bronze/cow_events",
    "Silver (Cows)": "s3a://lakehouse/silver/cows",
    "Gold (Herd Composition)": "s3a://lakehouse/gold/herd_composition",
    "Gold (Cow Lifecycle)": "s3a://lakehouse/gold/cow_lifecycle",
    "Gold (Daily Snapshots)": "s3a://lakehouse/gold/daily_snapshots"
}

for name, path in paths.items():
    try:
        # Try to read just the schema (doesn't load data)
        df = spark.read.format("delta").load(path)
        count = df.count()
        print(f"   ✅ {name}: {count} records")
    except Exception as e:
        if "Path does not exist" in str(e) or "NoSuchBucket" in str(e):
            print(f"   ⚠️  {name}: Not found")
        else:
            print(f"   ❌ {name}: Error - {str(e)[:80]}")

print("\n💡 If tables are missing, run the demo to populate data:")
print("   ./demo/run_demo_all.sh")

🔍 Checking MinIO container...
✅ MinIO container is running

📦 Ensuring buckets exist...
❌ Error accessing MinIO: Wrong FS: s3a://lakehouse/, expected: file:///

💡 Make sure you've run the demo to populate data:
   ./demo/run_demo_all.sh


IllegalArgumentException: Wrong FS: s3a://lakehouse/, expected: file:///

## ✅ Verify MinIO and Buckets
Check connectivity and ensure buckets exist

In [3]:
# Load Bronze layer
bronze = spark.read.format("delta").load("s3a://lakehouse/bronze/cow_events")

# Show schema
bronze.printSchema()

# Show recent events
bronze.orderBy("event_time", ascending=False).show(10, truncate=False)

26/01/28 19:45:34 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


Py4JJavaError: An error occurred while calling o52.load.
: org.apache.hadoop.fs.s3a.UnknownStoreException: s3a://lakehouse/bronze/cow_events/_delta_log
	at org.apache.hadoop.fs.s3a.S3AUtils.translateException(S3AUtils.java:257)
	at org.apache.hadoop.fs.s3a.S3AUtils.translateException(S3AUtils.java:170)
	at org.apache.hadoop.fs.s3a.S3AFileSystem.s3GetFileStatus(S3AFileSystem.java:3348)
	at org.apache.hadoop.fs.s3a.S3AFileSystem.innerGetFileStatus(S3AFileSystem.java:3185)
	at org.apache.hadoop.fs.s3a.S3AFileSystem.getFileStatus(S3AFileSystem.java:3053)
	at org.apache.hadoop.fs.FileSystem.exists(FileSystem.java:1760)
	at org.apache.hadoop.fs.s3a.S3AFileSystem.exists(S3AFileSystem.java:4263)
	at io.delta.storage.S3SingleDriverLogStore.listFromInternal(S3SingleDriverLogStore.java:201)
	at io.delta.storage.S3SingleDriverLogStore.listFrom(S3SingleDriverLogStore.java:306)
	at org.apache.spark.sql.delta.storage.LogStoreAdaptor.listFrom(LogStore.scala:452)
	at org.apache.spark.sql.delta.storage.

In [ ]:
# Filter weight events
bronze.filter("event_type = 'cow_weight_recorded'") \
      .select("event_id", "cow_id", "event_time", "payload") \
      .orderBy("event_time", ascending=False) \
      .show(5, truncate=False)

In [ ]:
# Event type distribution
bronze.groupBy("event_type").count().orderBy("count", ascending=False).show()

## 🥈 Silver Layer - Cleaned State (SCD Type 2)

In [ ]:
# Load Silver layer
silver = spark.read.format("delta").load("s3a://lakehouse/silver/cows")

# Show current state only
silver.filter("is_current = true") \
      .select("cow_id", "tag_number", "name", "breed", "weight_kg", "status", "valid_from") \
      .show(10, truncate=False)

In [ ]:
# Show full history for a specific cow (replace COW_ID)
cow_id = "<REPLACE_WITH_ACTUAL_COW_ID>"
silver.filter(f"cow_id = '{cow_id}'") \
      .select("name", "breed", "weight_kg", "status", "valid_from", "valid_to", "is_current") \
      .orderBy("valid_from") \
      .show(truncate=False)

In [ ]:
# Breed distribution (current state)
silver.filter("is_current = true") \
      .groupBy("breed").count() \
      .orderBy("count", ascending=False) \
      .show()

## 🥇 Gold Layer - Analytics

In [ ]:
# Load Gold herd composition
herd_comp = spark.read.format("delta").load("s3a://lakehouse/gold/herd_composition")
herd_comp.show(20, truncate=False)

In [ ]:
# Filter by dimension type
print("By Breed:")
herd_comp.filter("dimension_type = 'breed'").show()

print("\nBy Status:")
herd_comp.filter("dimension_type = 'status'").show()

print("\nBy Sex:")
herd_comp.filter("dimension_type = 'sex'").show()

In [ ]:
# Load Gold cow lifecycle
cow_lifecycle = spark.read.format("delta").load("s3a://lakehouse/gold/cow_lifecycle")
cow_lifecycle.select("cow_id", "current_breed", "current_status", "total_events", 
                     "days_in_system", "last_known_weight_kg") \
             .show(10, truncate=False)

In [ ]:
# Load Gold daily snapshots
daily_snapshots = spark.read.format("delta").load("s3a://lakehouse/gold/daily_snapshots")
daily_snapshots.orderBy("snapshot_date", ascending=False).show(10)

## 🔍 Advanced Queries

In [ ]:
# Time travel: Query Bronze as of 1 hour ago
from datetime import datetime, timedelta

one_hour_ago = (datetime.now() - timedelta(hours=1)).strftime("%Y-%m-%d %H:%M:%S")
bronze_historical = spark.read.format("delta") \
    .option("timestampAsOf", one_hour_ago) \
    .load("s3a://lakehouse/bronze/cow_events")

print(f"Events as of {one_hour_ago}: {bronze_historical.count()}")
print(f"Current events: {bronze.count()}")

In [ ]:
# Join Bronze and Silver to see event → state resolution
recent_events = bronze.filter("event_type = 'cow_created'") \
                      .select("cow_id", "event_time", "payload")

current_state = silver.filter("is_current = true") \
                      .select("cow_id", "name", "breed", "status")

recent_events.join(current_state, "cow_id") \
             .orderBy("event_time", ascending=False) \
             .show(5, truncate=False)

## 📊 Visualizations (Optional - requires matplotlib)

In [ ]:
# Uncomment if you want charts
# import matplotlib.pyplot as plt
# import pandas as pd

# # Breed distribution chart
# breed_df = herd_comp.filter("dimension_type = 'breed'").toPandas()
# plt.figure(figsize=(10, 6))
# plt.bar(breed_df['dimension_value'], breed_df['count'])
# plt.title('Herd Composition by Breed')
# plt.xlabel('Breed')
# plt.ylabel('Count')
# plt.xticks(rotation=45)
# plt.tight_layout()
# plt.show()